####Import Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 300)

RANDOM_STATE = 42

####Create Project Folders

In [2]:
folders = [
    "data/interim",
    "data/processed",
    "models",
    "outputs/figures",
    "outputs/reports",
    "outputs/metrics",
    "outputs/shap"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

####Load Dataset

In [4]:
DATA_PATH = "/content/amazon_reviews_cleaned.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset Shape: {df.shape}")
df.head(5)

Dataset Shape: (21055, 7)


,review_id,product_category,timestamp,country,rating,review,sentiment
0,REV-50BCBCD9,Sports,2024-09-16 13:44:26+00:00,US,1,"I registered on the website, tried to order a laptop, entered all the details, but instead of charging me and sending the product, they froze my account, demanding various verification documents. I sent them over. They said they would review them within 24 hours. In reality, it's been a week, an...",Negative
1,REV-6D2B2651,Toys,2024-09-16 18:26:46+00:00,GB,1,"Had multiple orders one turned up and driver had to phone as no door number on packaging, then waited all day for second package to get a message saying couldn't deliver as no number on packaging, 12 hours waiting in now don't even know when I'm getting delivery. Terrible will never use again",Negative
2,REV-F7E80372,Toys,2024-09-16 21:47:39+00:00,GB,1,"I informed these reprobates that I WOULD NOT BE IN as I was going to visit a sick relative, they told me they were going to send a OTP, I told them I could not receive it as I was travelling a long way on the underground, their reply was don’t worry we can text. I pointed out I can’t receive tex...",Negative
3,REV-ED2B173F,Sports,2024-09-17 07:15:49+00:00,AU,1,I have bought from Amazon before and no problems being very happy with the service and price. Amazon advertise the product at 61.23 us as soon as I logged in and tried to buy two of the items they were 86.75 for each.There is no way of contacting customer service I've spent an hour going round a...,Negative
4,REV-E48A7AB9,Fashion,2024-09-16 18:37:17+00:00,GB,1,If I could give a lower rate I would! I cancelled my Amazon Prime in February and subsequently found that they had continued to charge me. When I contacted them they refused to give details of who had set up the payment as I didn't have Prime membership at that time. My credit card company cance...,Negative


####Dataset Overview

In [6]:
df.info()
df.describe(include="all").T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21055 entries, 0 to 21054
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   review_id         21055 non-null  object
 1   product_category  21055 non-null  object
 2   timestamp         21055 non-null  object
 3   country           21054 non-null  object
 4   rating            21055 non-null  int64 
 5   review            21055 non-null  object
 6   sentiment         21055 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.1+ MB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
review_id,21055,21055,REV-4765654E,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_category,21055,7,Sports,3145,NaN,NaN,NaN,NaN,NaN,NaN,NaN
timestamp,21055,21054,2022-10-05 12:13:39+00:00,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,21054,148,US,9286,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rating,21055.0,NaN,NaN,NaN,2.186654,1.676769,1.0,1.0,1.0,4.0,5.0
review,21055,20407,Review text not found,630,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sentiment,21055,3,Negative,14350,NaN,NaN,NaN,NaN,NaN,NaN,NaN


####Missing Values

In [7]:
missing = (
    df.isnull()
      .sum()
      .reset_index()
)

missing.columns = ["Column", "Missing Values"]

missing["Percentage"] = (
    missing["Missing Values"] /
    len(df)
) * 100

missing

,Column,Missing Values,Percentage
0,review_id,0,0.000000
1,product_category,0,0.000000
2,timestamp,0,0.000000
3,country,1,0.004749
4,rating,0,0.000000
5,review,0,0.000000
6,sentiment,0,0.000000


####Duplicate Records

In [8]:
duplicates = df.duplicated().sum()

print(f"Duplicate Rows: {duplicates}")

Duplicate Rows: 0


####Convert Data Types

In [9]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce",
    utc=True
)

#####Create

In [10]:
df["date"] = df["timestamp"].dt.date
df["year"] = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month
df["month_name"] = df["timestamp"].dt.month_name()
df["weekday"] = df["timestamp"].dt.day_name()
df["hour"] = df["timestamp"].dt.hour

####Missing Value Treatment

In [11]:
df = df.dropna(
    subset=[
        "review",
        "rating",
        "country",
        "sentiment"
    ]
)

####Review Basic Statistics

In [12]:
print(df["sentiment"].value_counts())

print(df["country"].nunique())

print(df["product_category"].nunique())

sentiment
Negative    14350
Positive     5819
Neutral       885
Name: count, dtype: int64
148
7


####Save Clean Dataset

In [13]:
df.to_csv(
    "data/interim/clean_reviews.csv",
    index=False
)

print("Clean dataset saved.")

Clean dataset saved.
